In [ ]:
!unzip morse_dataset.zip "morse_dataset/*" -d extracted_morse_dataset

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import librosa
import random
from sklearn.model_selection import train_test_split
from tqdm import tqdm

############################################
# 1) Алфавит
############################################
alphabet = {
    '<pad>': 0,
    ' ': 1,
    '#': 2,
    '0': 3,
    '1': 4,
    '2': 5,
    '3': 6,
    '4': 7,
    '5': 8,
    '6': 9,
    '7': 10,
    '8': 11,
    '9': 12,
    'А': 13,
    'Б': 14,
    'В': 15,
    'Г': 16,
    'Д': 17,
    'Е': 18,
    'Ж': 19,
    'З': 20,
    'И': 21,
    'Й': 22,
    'К': 23,
    'Л': 24,
    'М': 25,
    'Н': 26,
    'О': 27,
    'П': 28,
    'Р': 29,
    'С': 30,
    'Т': 31,
    'У': 32,
    'Ф': 33,
    'Х': 34,
    'Ц': 35,
    'Ч': 36,
    'Ш': 37,
    'Щ': 38,
    'Ъ': 39,
    'Ы': 40,
    'Ь': 41,
    'Э': 42,
    'Ю': 43,
    'Я': 44
}
num_classes = len(alphabet)  # 45 (0..44, 0=blank)

############################################
# 2) encode_label
############################################
def encode_label(text: str, alpha: dict) -> list:
    indices = []
    for ch in text:
        if ch in alpha:
            indices.append(alpha[ch])
    return indices

############################################
# 3) Аудио-аугментация (до Mel)
############################################
def augment_waveform(y, sr=16000, noise_factor=0.003, time_stretch_range=(0.9,1.1)):
    """
    1) Add random noise
    2) Time stretch
    """
    # add noise
    if random.random() < 0.5:
        noise = np.random.randn(len(y)) * noise_factor
        y = y + noise.astype(y.dtype)
    # time stretch
    if random.random() < 0.5:
        rate = random.uniform(*time_stretch_range)
        try:
            y = librosa.effects.time_stretch(y, rate=rate)
        except:
            pass
    return y

############################################
# 4) SpecAugment (для Mel)
############################################
def spec_augment(mel, max_freq_mask=10, max_time_mask=20):
    """
    Маскируем случайную полосу частот + времени
    mel: [n_mels, time]
    """
    n_mels, n_time = mel.shape
    # freq mask
    if random.random() < 0.5:
        fsize = random.randint(1, max_freq_mask)
        f0 = random.randint(0, n_mels - fsize)
        mel[f0:f0+fsize, :] = 0.0
    # time mask
    if random.random() < 0.5:
        tsize = random.randint(1, max_time_mask)
        t0 = random.randint(0, n_time - tsize)
        mel[:, t0:t0+tsize] = 0.0
    return mel

############################################
# 5) Преобразование audio -> Mel
############################################
def audio_to_melspectrogram(y, sr=16000, n_mels=64, n_fft=1024, hop_length=512,
                            do_specaug=False):
    S = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels
    )
    log_S = librosa.power_to_db(S, ref=np.max)
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)

    # если хотим, применяем SpecAug
    if do_specaug and random.random() < 0.7:
        log_S_norm = spec_augment(log_S_norm, max_freq_mask=10, max_time_mask=20)

    return log_S_norm  # [n_mels, time]

############################################
# 6) Наш Dataset
############################################
class MorseAudioCTCDataset(Dataset):
    def __init__(self, df, sr=16000, augment=False, transform=None):
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.augment = augment     # если True, применяем augment_waveform
        self.transform = transform # функция (y)->Mel

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        audio_path = "morse_dataset/" + row['id']
        y, _ = librosa.load(audio_path, sr=self.sr)

        if self.augment:
            y = augment_waveform(y, sr=self.sr)

        if self.transform:
            mel = self.transform(y, sr=self.sr)
        else:
            mel = y

        mel_tensor = torch.tensor(mel, dtype=torch.float)
        # label
        text_label = row['message']
        label_indices = encode_label(text_label, alphabet)
        label_tensor = torch.tensor(label_indices, dtype=torch.long)

        time_dim = mel_tensor.shape[1]
        return mel_tensor, label_tensor, time_dim

############################################
# 7) ctc_collate_fn
############################################
def ctc_collate_fn(batch, pool_time_factor=4):
    mel_list = []
    label_list = []
    tgt_len_list = []
    raw_time_list = []

    for (mel, label, tdim) in batch:
        mel_list.append(mel)
        label_list.append(label)
        tgt_len_list.append(len(label))
        raw_time_list.append(tdim)

    max_time = max(m.shape[1] for m in mel_list)
    padded_mels = []
    for mel in mel_list:
        diff = max_time - mel.shape[1]
        if diff>0:
            mel = F.pad(mel, (0,diff), value=0.0)
        mel = mel.unsqueeze(0) # => [1, n_mels, max_time]
        padded_mels.append(mel)

    audio_batch = torch.stack(padded_mels, dim=0) # [B, 1, n_mels, max_time]
    labels_concat = torch.cat(label_list, dim=0)

    input_lengths = []
    for rt in raw_time_list:
        input_lengths.append(rt // pool_time_factor)
    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor(tgt_len_list, dtype=torch.long)

    return audio_batch, labels_concat, input_lengths, target_lengths

############################################
# 8) Увеличенная модель
############################################
class EnhancedCTCModel(nn.Module):
    """
    Два Conv+Pool -> BiLSTM -> Linear
    Теперь hidden_size=256, можно выставить lstm_layers=3
    """
    def __init__(self, num_classes=45, in_channels=1, n_mels=64,
                 hidden_size=256, lstm_layers=3, dropout=0.3):
        super(EnhancedCTCModel, self).__init__()

        # Блок 1: Conv -> BN -> ReLU -> Pool
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )
        # Блок 2: Conv -> BN -> ReLU -> Pool
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )
        # => freq/time /4, channels=32 => LSTM input=32*(n_mels//4)
        self.lstm = nn.LSTM(
            input_size=32 * (n_mels//4),
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            dropout=(dropout if lstm_layers>1 else 0.0),
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size*2, num_classes)

    def forward(self, x):
        """
        x: [B, 1, n_mels=64, time]
        => [time//4, B, num_classes]
        """
        x = self.conv1(x)
        x = self.conv2(x)
        b,c,f,t = x.shape
        x = x.view(b, c*f, t)
        x = x.permute(2,0,1)  # => [T, B, feature]
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits  # [T, B, num_classes]

############################################
# 9) Цикл обучения
############################################
def train_ctc_loop(model, train_loader, val_loader, num_epochs=15, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        model.train()
        train_loss_sum = 0.0
        train_bar = tqdm(train_loader, desc="Train", leave=False)

        for audio_batch, labels_concat, input_lengths, target_lengths in train_bar:
            audio_batch = audio_batch.to(device)
            labels_concat = labels_concat.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            logits = model(audio_batch)  # [T, B, C]
            log_probs = F.log_softmax(logits, dim=2)
            loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss_sum / len(train_loader)

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_bar = tqdm(val_loader, desc="Val", leave=False)
        with torch.no_grad():
            for audio_batch, labels_concat, input_lengths, target_lengths in val_bar:
                audio_batch = audio_batch.to(device)
                labels_concat = labels_concat.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits = model(audio_batch)
                log_probs = F.log_softmax(logits, dim=2)
                loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
                val_loss_sum += loss.item()
                val_bar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss_sum / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

############################################
# 10) Main
############################################
if __name__ == "__main__":
    df = pd.read_csv("train.csv")
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    # 1) Для train применим augment=True, а также do_specaug=True
    def transform_train(y, sr=16000):
        return audio_to_melspectrogram(y, sr=sr, do_specaug=True)

    # Для val ничего не меняем
    def transform_val(y, sr=16000):
        return audio_to_melspectrogram(y, sr=sr, do_specaug=False)

    train_dataset = MorseAudioCTCDataset(
        train_df, sr=16000, augment=True, transform=transform_train
    )
    val_dataset = MorseAudioCTCDataset(
        val_df, sr=16000, augment=False, transform=transform_val
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=4,
        shuffle=True,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=4,
        shuffle=False,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )

    # Модель чуть больше: hidden_size=256, 2-3 слоев LSTM
    model = EnhancedCTCModel(
        num_classes=45,
        in_channels=1,
        n_mels=64,
        hidden_size=256,
        lstm_layers=3,  # можно 3, если хватит ресурсов
        dropout=0.3
    )

    # Проверим shapes
    audio_batch, labels_concat, input_lengths, target_lengths = next(iter(train_loader))
    print("audio_batch:", audio_batch.shape)
    print("labels_concat:", labels_concat.shape, labels_concat[:10])
    print("input_lengths:", input_lengths)
    print("target_lengths:", target_lengths)

    # Обучение
    train_ctc_loop(
        model,
        train_loader,
        val_loader,
        num_epochs=32,
        lr=1e-3
    )


audio_batch: torch.Size([4, 1, 64, 251])
labels_concat: torch.Size([41]) tensor([39, 31, 25,  3, 20, 10,  4, 29, 24, 19])
input_lengths: tensor([62, 57, 62, 61])
target_lengths: tensor([ 7, 12, 13,  9])

Epoch 1/32


Epoch 1/32 | Train Loss: 4.0203 | Val Loss: 3.9974

Epoch 2/32


Epoch 2/32 | Train Loss: 3.9836 | Val Loss: 3.9684

Epoch 3/32


Epoch 3/32 | Train Loss: 3.9658 | Val Loss: 3.9575

Epoch 4/32


Epoch 4/32 | Train Loss: 3.9620 | Val Loss: 3.9616

Epoch 5/32


...

Epoch 28/32 | Train Loss: 0.5022 | Val Loss: 0.2621

Epoch 29/32


Epoch 29/32 | Train Loss: 0.4902 | Val Loss: 0.2594

Epoch 30/32


Epoch 30/32 | Train Loss: 0.4948 | Val Loss: 0.2787

Epoch 31/32


Epoch 31/32 | Train Loss: 0.4867 | Val Loss: 0.2442

Epoch 32/32


Epoch 32/32 | Train Loss: 0.4787 | Val Loss: 0.2636


In [2]:
torch.save(model.state_dict(), "Bigger_one_2.pth")


In [3]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import pandas as pd

#########################################
# Алфавит
#########################################
alphabet = {
    '<pad>': 0,
    ' ': 1,
    '#': 2,
    '0': 3,
    '1': 4,
    '2': 5,
    '3': 6,
    '4': 7,
    '5': 8,
    '6': 9,
    '7': 10,
    '8': 11,
    '9': 12,
    'А': 13,
    'Б': 14,
    'В': 15,
    'Г': 16,
    'Д': 17,
    'Е': 18,
    'Ж': 19,
    'З': 20,
    'И': 21,
    'Й': 22,
    'К': 23,
    'Л': 24,
    'М': 25,
    'Н': 26,
    'О': 27,
    'П': 28,
    'Р': 29,
    'С': 30,
    'Т': 31,
    'У': 32,
    'Ф': 33,
    'Х': 34,
    'Ц': 35,
    'Ч': 36,
    'Ш': 37,
    'Щ': 38,
    'Ъ': 39,
    'Ы': 40,
    'Ь': 41,
    'Э': 42,
    'Ю': 43,
    'Я': 44
}
idx2char = {v: k for k, v in alphabet.items()}

#########################################
# Модель (точно такая же, как при обучении)
#########################################
#########################################
# Используем правильную модель
#########################################
class EnhancedCTCModel(nn.Module):
    """
    Два Conv+Pool -> BiLSTM -> Linear
    Теперь hidden_size=256, можно выставить lstm_layers=3
    """
    def __init__(self, num_classes=45, in_channels=1, n_mels=64,
                 hidden_size=256, lstm_layers=3, dropout=0.3):
        super(EnhancedCTCModel, self).__init__()

        # Блок 1: Conv -> BN -> ReLU -> Pool
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )
        # Блок 2: Conv -> BN -> ReLU -> Pool
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )
        # => freq/time /4, channels=32 => LSTM input=32*(n_mels//4)
        self.lstm = nn.LSTM(
            input_size=32 * (n_mels//4),
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            dropout=(dropout if lstm_layers>1 else 0.0),
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size*2, num_classes)

    def forward(self, x):
        """
        x: [B, 1, n_mels=64, time]
        => [time//4, B, num_classes]
        """
        x = self.conv1(x)
        x = self.conv2(x)
        b,c,f,t = x.shape
        x = x.view(b, c*f, t)
        x = x.permute(2,0,1)  # => [T, B, feature]
        lstm_out, _ = self.lstm(x)
        logits = self.fc(lstm_out)
        return logits  # [T, B, num_classes]

#########################################
# Превращение аудио -> Mel
#########################################
def audio_to_melspectrogram(y, sr=16000, n_mels=64, n_fft=1024, hop_length=512):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, 
                                       hop_length=hop_length, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)
    return log_S_norm

#########################################
# Greedy decode (argmax)
#########################################
def ctc_greedy_decode(logits, blank=0):
    """
    logits: [T, 1, C]
    Возвращаем список индексов (без повторяющихся подряд, без blank).
    """
    # [T, 1, C] -> argmax => [T, 1]
    argmax = torch.argmax(logits, dim=2)  # [T, 1]
    argmax = argmax.squeeze(1).cpu().numpy().tolist()  # [T]

    decoded = []
    prev = None
    for idx in argmax:
        if idx != blank and idx != prev:
            decoded.append(idx)
        prev = idx
    return decoded

#########################################
# Инференс + submit
#########################################
def infer_greedy(
    model_path="Bigger_one_2.pth",
    test_csv="test.csv",
    audio_dir="morse_dataset",
    output_csv="submit_6.csv"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # 1) Загружаем модель с правильной конфигурацией
    model = EnhancedCTCModel(
        num_classes=45,
        in_channels=1,
        n_mels=64,
        hidden_size=256,
        lstm_layers=3,         # ✅ обязательно 3, как при обучении
        dropout=0.3
    )
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    # 2) Читаем test.csv
    df_test = pd.read_csv(test_csv)
    results = []

    # 3) Пробегаем по всем аудиофайлам
    for idx, row in df_test.iterrows():
        file_id = row['id']
        audio_path = os.path.join(audio_dir, file_id)

        # Аудио -> Mel
        y, _ = librosa.load(audio_path, sr=16000)
        mel = audio_to_melspectrogram(y, sr=16000)  # [n_mels=64, time]
        mel_tensor = torch.tensor(mel, dtype=torch.float).unsqueeze(0).unsqueeze(0).to(device)
        # => [1, 1, n_mels, time]

        # Прогоняем через модель
        with torch.no_grad():
            logits = model(mel_tensor)  # [T, 1, 45]
            log_probs = F.log_softmax(logits, dim=2)

        # Greedy decode
        pred_indices = ctc_greedy_decode(log_probs, blank=0)
        pred_text = ''.join(idx2char[i] for i in pred_indices if i in idx2char)

        results.append({
            "id": file_id,
            "message": pred_text
        })

    # 6) Сохраняем предсказания
    df_sub = pd.DataFrame(results)
    df_sub.to_csv(output_csv, index=False)
    print(f"✅ Saved submission: {output_csv}")


#########################################
# Запуск
#########################################
if __name__ == "__main__":
    infer_greedy(
        model_path="Bigger_one_2.pth",
        test_csv="test.csv",
        audio_dir="morse_dataset",
        output_csv="submission_6.csv"
    )

✅ Saved submission: submission_6.csv


In [5]:
sub = pd.read_csv("submission_6.csv")

In [6]:
sub.tail(17)

,id,message
4983,34984.opus,ДАМИСАМТДОТИРСЫСАМЦ10
4984,34985.opus,ИЛ ЬСВЕДТКЧВНТИ ЯМДМЫМЮНЯМЦ ЮТИЫМ ЬТКТЧМЫН ДТЫ...
4985,34986.opus,АНХ ВСОЕГЬ Р ВКТДАМФНКШМДНИ ГКНЖТА2
4986,34987.opus,ИЛ ДАСНП ЬСОЕМУНТИЛГВКСОЕП ЬКСХЫСУС ХНУЮН ХНУСИ
4987,34988.opus,ТЬТКП ДСЯНКМЫОЦ ИМК 22
4988,34989.opus,КНОЬКМ СОЛЫМОП ЬСЮНЮМ
4989,34990.opus,ИЛЬСОЕНДМЫМ ЙЫРУГЙЫМЧАТУС ЬКТДЛХТ #К4АЛШ ДЛУСВ...
4990,34991.opus,ИЛ ДАСДП ОСЮВНЫМ НЮЙГРГ ТЫТУКНЩН
4991,34992.opus,АСЯТ ИСЧД МГЖХПОЦ ЮС НЙЮСЫЗЕАЗДТКАСВЕП2МИДСЫС
4992,34993.opus,ОЫЗДЛ ОЫЛ5ВТ #ЕС ЬСЫНСТ ЕСРЫЪНАМЕ#П
